<a href="https://colab.research.google.com/github/Terry4715/MVO-backtest/blob/main/CodeAlong_Module4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import sys

from google.colab import drive


# Mount Google Drive
drive.mount('/content/drive/')

# Define the path to directory
data_path = '/content/drive/MyDrive/ColabNotebooks/EDHEC_Learning'

# Add the directory to sys.path to allow Python to find custom module
if data_path not in sys.path:
    sys.path.append(data_path)

# Import your custom module
import toolkit as tk

Mounted at /content/drive/
Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


# Present Value of Liabilities and Funding Ratio

The present value of a set of liabilities $L$ where each liability $L_i$ is due at time $t_i$ is give by:

$$ PV(L) = \sum_{i=1}^{k} B(t_i) L_i$$

where $B(t_i)$ is the price of a pure discount bond that pays 1 dollar at time $t_i$

If we assume the yield curve is flat and the annual rate of interest is $r$ then $B(t)$ is given by

$$B(t) = \frac{1}{(1+r)^t}$$


In [2]:
def discount(t, r):
  """
  discount factor that computes the price of a pure discount bond that pays a dollar at time t, given interest rate r
  """
  return 1/(1+r)**t

In [3]:
discount(4.5, 0.03)

0.8754523015810893

In [4]:
def pv(l, r):
  """
  computes the present value of a sequence of liabilities
  l is indexed by the time, and the values are the amounts of each liability
  r is the interest rate
  returns the present value of the sequence
  """
  dates = l.index
  discounts = discount(dates, r)
  return (discounts * l).sum()

In [5]:
liabilities = pd.Series(data=[1, 1.5, 2, 2.5], index=[3, 3.5, 4, 4.5])
pv(liabilities, 0.03)

np.float64(6.233320315080044)

In [6]:
6.233320315080044 * (1.03**4.5)

7.120114144222943

In [7]:
def funding_ratio(assets, liabilities, r):
  """
  computes the funding ratio of some assets given liabilities and interest rate
  """
  return assets/pv(liabilities, r)

In [8]:
funding_ratio(5, liabilities, 0.03)

np.float64(0.8021407126958778)

In [9]:
funding_ratio(5, liabilities, 0.02)

np.float64(0.7720304366941647)

In [10]:
funding_ratio(5, liabilities, 0.05)

np.float64(0.8649082768407929)

In [11]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

In [12]:
def show_funding_ratio(assets, r):
  fr = funding_ratio(assets, liabilities, r)
  print(f'{fr*100:.2f}')


controls = widgets.interactive(show_funding_ratio,
                               assets = widgets.IntSlider(min=1, max=10, step=1, value=5),
                               r = (0, 0.2, 0.01))
display(controls)

interactive(children=(IntSlider(value=5, description='assets', max=10, min=1), FloatSlider(value=0.1, descript…

### short rate Vs annualsied
If we compound every 1/N of a year:
$$\left(1+\frac{r}{N}\right)^N$$

In [13]:
(1 + 1/2)**2

2.25

In [14]:
def inst_to_ann(r):
  """
  Converts short rate to an annualised rate
  """
  return np.expm1(r)

def ann_to_inst(r):
  """
  Converts an annualised rate to short rate
  """
  return np.log1p(r)

### CIR Model to simulate changes in Interest Rates and Liability Hedging
$$ dr_{t}=a(b-r_{t})\,dt+\sigma {\sqrt {r_{t}}}\,dW_{t} $$